In [ ]:
import math
import numpy as np
from SDP_utils.combinatorics import occupation_index, nu_set_iterator, multinomial, nu_iterator, dim_sym_kd
from SDP_utils.isotopic_proj import isotypic_projector
from tqdm.notebook import tqdm

SDP_TOL = 10**(-7)

In [51]:
dim_sym_kd(2, 16)

136

In [52]:
816**2/ 132**2

38.21487603305785

In [49]:
9**3

729

In [2]:
def antiSymProjector(k):
    return [1] * k

In [3]:
def V_builder(k:int,d:int) -> np.ndarray:
    V = np.zeros((math.comb(k + d - 1, k), d**k))
    for nu in nu_iterator(k,d):
        fact = 1.0 / math.sqrt(multinomial(k, nu))
        nu_index = occupation_index(nu, k, d)
        V[nu_index, list(nu_set_iterator(nu))] = fact
    return V    

# print(V_builder(3, 2))

In [4]:
def V_l_V_kl_builder(k:int,d:int,l:int) -> np.ndarray:
    Vl = V_builder(l,d)
    Vkl = V_builder(k-l,d)
    return np.kron(Vl, Vkl)

def W_l_builder(k:int,d:int,l:int) -> np.ndarray:
    return V_l_V_kl_builder(k,d,l) @ V_builder(k,d).transpose()

In [5]:
def isotopyc_I_perm(m:int, n:int, k:int) -> list[int]:

    ret: list[int] = []

    def recA(new_index:int, dim_num:int):
        if dim_num >= k + 1:
            recB(new_index, 1)
            return
        
        stride = (n*m)**(k-dim_num) * n
        for i in range(new_index, new_index + m * stride, stride):
            recA(i, dim_num + 1)
    
    def recB(new_index:int, dim_num:int):
    
        if dim_num >= k + 1:
            ret.append(new_index)
            return
        
        stride = (n*m)**(k-dim_num)
        for i in range(new_index, new_index + n*stride, stride):
            recB(i, dim_num + 1)

    recA(0, 1)
    return ret

def isotypic_ot_I(m, n, k, lam) -> np.ndarray:
    Pi_lambda = isotypic_projector(lam, m, k)
    I = np.identity(n**k)
    M = np.kron(Pi_lambda, I)            # ordered A_1..A_k B_1..B_k

    perm = np.asarray(isotopyc_I_perm(m, n, k))
    inv  = np.argsort(perm)
    return M[np.ix_(inv, inv)]           # now ordered A_1 B_1 ... A_k B_k



In [6]:
def alpha_dag_j_builder(k:int, d:int, j:int):
    alpha_dag_j = np.zeros((dim_sym_kd(k,d), dim_sym_kd(k-1,d)))
    for nu_darrow in nu_iterator(k-1,d):
        index_V_darrow = occupation_index(nu_darrow, k-1, d)
        nu_darrow[j] += 1
        index_V_darrow_e_j = occupation_index(nu_darrow, k, d)
        alpha_dag_j[index_V_darrow_e_j,index_V_darrow] = math.sqrt(nu_darrow[j])
    return alpha_dag_j

print(alpha_dag_j_builder(3,3,2))

[[0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [1.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.         0.        ]
 [0.         0.         1.41421356 0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         1.         0.         0.        ]
 [0.         0.         0.         0.         1.41421356 0.        ]
 [0.         0.         0.         0.         0.         1.73205081]]


```
conda activate sage
pip install qics
```

In [7]:
from numpy import ndarray
import picos

def SDP(lam: list[int], rho: np.ndarray, m: int, n: int, solver="qics"):
    k = sum(lam)
    d, _ = rho.shape
    assert d == m * n

    sym_d: int   = dim_sym_kd(k, d)
    V   = V_builder(k, d)
    a_dag = [alpha_dag_j_builder(k, d, j) for j in range(d)]
    AdA = [[picos.Constant(Ad @ A.T) for A in a_dag] for Ad in a_dag]
    VPi = picos.Constant(V @ isotypic_ot_I(m, n, k, lam) @ V.T)
    Wls = [(picos.Constant(W_l_builder(k, d, l)), dim_sym_kd(l, d), dim_sym_kd(k - l, d))
       for l in range(1, k//2 + 1)]


    P = picos.Problem()
    omega_sym = picos.HermitianVariable("omega_sym", sym_d)

    # objective:  min tr( (V Pi^lam(x)I V^dag) omega_sym )
    P.set_objective("min", picos.trace(VPi * omega_sym).real) # type: ignore

    # (1) PSD
    P.add_constraint(omega_sym >> 0)

    #(2) marginal:  tr_{!=1}(omega)_{j,j'} = (1/k) tr( a_{j'}^dag a_j  omega ) = rho
    marg = picos.block([[ (1/k) * picos.trace(AdA[jp][j] * omega_sym) for jp in range(d)] for j in range(d)]) # type: ignore
    P.add_constraint(marg == picos.Constant("rho", rho))

    # (3) PPT on the l | k-l cuts,  l = 1 .. floor(k/2)
    for Wl, dl, dkl in Wls:
        B = Wl * omega_sym * Wl.T
        P.add_constraint(B.partial_transpose(subsystems=0, dimensions=(dl, dkl)) >> 0)  # type: ignore

    P.solve(solver=solver)
    return P.value, omega_sym.value


In [8]:
import numpy as np
import picos

def SDP_full(lam: list[int], rho: np.ndarray, m: int, n: int, solver="qics"):
    """Schmidt-number SDP on the full (C^d)^{ot k} space — no symmetric reduction.

    Variable: omega_{1..k} in C^{d^k x d^k}, d = m*n (each copy is one A_iB_i system).
    """
    k = sum(lam)
    d, _ = rho.shape
    assert d == m * n

    Dk = d ** k
    PiI    = picos.Constant("PiI",    isotypic_ot_I(m, n, k, lam))   # Pi^lam (x) I, order A_1B_1...A_kB_k
    Pi_sym = picos.Constant("Pi_sym", V_builder(k, d).T @ V_builder(k, d))  # = V^dag V
    rho_c  = picos.Constant("rho", rho)

    P = picos.Problem()
    omega = picos.HermitianVariable("omega", Dk)

    # objective:  min tr( (Pi^lam (x) I) omega )
    P.set_objective("min", picos.trace(PiI * omega).real)  # type: ignore

    # (1) PSD
    P.add_constraint(omega >> 0)

    # (2) supported on the symmetric subspace:  Pi_sym omega Pi_sym = omega
    P.add_constraint(Pi_sym * omega * Pi_sym == omega)

    # (3) marginal:  tr_{!=1}(omega) = rho   (keep copy 0, trace out copies 1..k-1)
    P.add_constraint(
        omega.partial_trace(subsystems=list(range(1, k)), dimensions=[d] * k) == rho_c)  # type: ignore

    # (4) PPT on the cuts S = {1,...,l},  l = 1 .. floor(k/2)
    for l in range(1, k // 2 + 1):
        P.add_constraint(
            omega.partial_transpose(subsystems=list(range(l)), dimensions=[d] * k) >> 0)  # type: ignore

    P.solve(solver=solver)
    return P.value, omega.value

$F |\phi^+\rangle\langle\phi^+| + (1-F)(I - |\phi^+\rangle\langle\phi^+|)/(d^2-1) $

In [19]:
def Cm_Cm_isotropic_state(m:int, F:float) -> np.ndarray:
    d: int = m*m
    max_entangled = np.zeros((d, 1), dtype=complex)
    for i in range(m):
        max_entangled[i * m + i, 0] = 1.0 / np.sqrt(d)
    phi_plus_M = max_entangled @ max_entangled.conj().T
    rho_f = F * phi_plus_M  + (1-F) * (np.identity(d, dtype=complex) - phi_plus_M)/(d - 1)
    return rho_f

Cm_Cm_isotropic_state(2, 0.5)


array([[0.25      +0.j, 0.        +0.j, 0.        +0.j, 0.08333333+0.j],
       [0.        +0.j, 0.16666667+0.j, 0.        +0.j, 0.        +0.j],
       [0.        +0.j, 0.        +0.j, 0.16666667+0.j, 0.        +0.j],
       [0.08333333+0.j, 0.        +0.j, 0.        +0.j, 0.25      +0.j]])

In [40]:
for f10 in range(10):
    f: float = f10 / 10
    isof = Cm_Cm_isotropic_state(2, f)
    obj, omega_sym = SDP([3,1], isof, 2,2)
    if obj < SDP_TOL:
        print(f"SN < {2}, obj = {obj} == 0, f = {f}")
    else:
        print(f"SN >= {2}, obj = {obj} != 0, f = {f}")

    isof = Cm_Cm_isotropic_state(2, f)
    obj, omega_sym = SDP_full(antiSymProjector(2), isof, 2,2)
    if obj < SDP_TOL:
        print(f"SN < {2}, obj = {obj} == 0, f = {f}")
    else:
        print(f"SN >= {2}, obj = {obj} != 0, f = {f}")
    print()

SN < 2, obj = -5.582972325765545e-09 == 0, f = 0.0
SN < 2, obj = 2.586280495320281e-10 == 0, f = 0.0

SN < 2, obj = -5.259413752602138e-09 == 0, f = 0.1
SN < 2, obj = 5.307675514376609e-10 == 0, f = 0.1

SN < 2, obj = -3.370005585093955e-08 == 0, f = 0.2
SN < 2, obj = -1.9937797940405844e-11 == 0, f = 0.2

SN < 2, obj = -3.9501672945661814e-08 == 0, f = 0.3
SN < 2, obj = -1.447895622841422e-11 == 0, f = 0.3

SN < 2, obj = -2.531690958436361e-09 == 0, f = 0.4
SN < 2, obj = 3.638935021366141e-10 == 0, f = 0.4

SN < 2, obj = -1.3060381309423774e-08 == 0, f = 0.5
SN < 2, obj = -6.14577635310587e-10 == 0, f = 0.5

SN < 2, obj = -2.529091227242153e-08 == 0, f = 0.6
SN < 2, obj = -1.8527878883900506e-10 == 0, f = 0.6

SN >= 2, obj = 0.010659612933759582 != 0, f = 0.7
SN >= 2, obj = 0.003571427487122747 != 0, f = 0.7

SN >= 2, obj = 0.06228586067183216 != 0, f = 0.8
SN >= 2, obj = 0.021491223623046778 != 0, f = 0.8

SN >= 2, obj = 0.1593158144919668 != 0, f = 0.9
SN >= 2, obj = 0.0593137252680

In [ ]:
m = 3
for f10 in range(10):
    f: float = f10 / 10
    isof = Cm_Cm_isotropic_state(m, f)
    obj, omega_sym = SDP(antiSymProjector(2), isof, m,m)
    if obj < SDP_TOL:
        print(f"SN < {2}, obj = {obj} == 0, f = {f}")
    else:
        print(f"SN >= {2}, obj = {obj} != 0, f =y {f}")


SN >= 2, obj = 0.11111107746797627 != 0, f =y 1.0


In [ ]:
# f: float = f10 / 10
# isof = Cm_Cm_isotropic_state(m, f)
# obj, omega_sym = SDP(antiSymProjector(3), isof, m,m)
# if obj < SDP_TOL:
#     print(f"SN < {2}, obj = {obj} == 0, f = {f}")
# else:
#     print(f"SN >= {2}, obj = {obj} != 0, f =y {f}")

In [ ]:
# m = 4
# for f10 in range(5,7):
#     f: float = f10 / 10
#     isof = Cm_Cm_isotropic_state(m, f)
#     obj, omega_sym = SDP(antiSymProjector(2), isof, m,m)
#     if obj < SDP_TOL:
#         print(f"SN < {2}, obj = {obj} == 0, f = {f}")
#     else:
#         print(f"SN >= {2}, obj = {obj} != 0, f = {f}")

KeyboardInterrupt: 

# Random entangled mixed state in $\mathbb{C}^2 \otimes \mathbb{C}^2$

In [27]:
def random_density_matrix(n=4):
    """Bures or Hilbert-Schmidt random density matrix."""
    # Ginibre ensemble: random complex matrix
    G = np.random.randn(n, n) + 1j * np.random.randn(n, n)
    rho = G @ G.conj().T
    return rho / np.trace(rho)

def is_entangled_ppt(rho):
    """
    PPT (Peres-Horodecki) criterion — necessary for separability.
    If partial transpose has a negative eigenvalue → entangled.
    (For 2x2 systems this is also sufficient.)
    """
    # Partial transpose on second subsystem
    rho_r = rho.reshape(2, 2, 2, 2)
    pt = rho_r.transpose(0, 3, 2, 1).reshape(4, 4)
    eigvals = np.linalg.eigvalsh(pt)
    return np.any(eigvals < -1e-10)

def random_mixed_entangled_state():
    while True:
        rho = random_density_matrix()
        if is_entangled_ppt(rho):
            return rho

In [42]:
m = 2
nb_identified = 0
for i in range(100):
    state = random_mixed_entangled_state()
    obj, omega_sym = SDP(antiSymProjector(2), state, m,m)
    obj2, omega_sym = SDP_full(antiSymProjector(2), state, m,m)
    print(F"{obj}, {obj2}, {obj/ obj2}")
    # print(obj)
print(f"Correctly identified {nb_identified} of 100 in C2xC2")

0.0020559626335506197, 0.0020559626342204138, 0.9999999996742187
0.01175621472984057, 0.011756214737083215, 0.9999999993839306
0.0017875400092560792, 0.0017875400076109715, 1.0000000009203194
0.000510839077619351, 0.0005108390778335303, 0.9999999995807303
0.008370244999013718, 0.008370245028097526, 0.9999999965253337
0.0002812196404526103, 0.0002812196413747199, 0.9999999967210341
0.012043240391445102, 0.012043240399937507, 0.9999999992948405
0.001779931215117908, 0.001779931222312181, 0.9999999959581174
0.01066754199639654, 0.010667542030673605, 0.999999996786789
0.02330354436123732, 0.02330354435886816, 1.0000000001016653
0.008970651101733448, 0.00897065114738443, 0.9999999949110736
0.023908319569535555, 0.02390831957561239, 0.9999999997458277
0.03814739451878163, 0.03814739447262133, 1.0000000012100512
0.0015346631561841337, 0.0015346631561407899, 1.0000000000282432
0.00017790611869277287, 0.0001779061185270825, 1.0000000009313361
8.905364065728283e-05, 8.90536430139463e-05, 0.99999

# Entangled mixed state in $\mathbb{C}^3 \otimes \mathbb{C}^3$

In [ ]:
import numpy as np
from scipy.stats import unitary_group

rng = np.random.default_rng()

# --- certificates (optional, for checking) ---
def is_ppt(rho, dA=3, dB=3, tol=1e-10):
    M = rho.reshape(dA,dB,dA,dB).transpose(0,3,2,1).reshape(dA*dB, dA*dB)
    return np.linalg.eigvalsh((M+M.conj().T)/2).min() > -tol

def realignment_norm(rho, dA=3, dB=3):          # >1 certifies entanglement (CCNR)
    R = rho.reshape(dA,dB,dA,dB).transpose(0,2,1,3).reshape(dA*dA, dB*dB)
    return np.linalg.svd(R, compute_uv=False).sum()

# --- (1) Horodecki one-parameter family: bound entangled for 0 < a < 1 ---
def horodecki_state(a):
    b = 8*a + 1
    r = np.zeros((9,9), dtype=complex)
    for i in (0,1,2,3,4,5,7): r[i,i] = a
    r[6,6] = r[8,8] = (1+a)/2
    for i,j in [(0,4),(4,0),(0,8),(8,0),(4,8),(8,4)]: r[i,j] = a
    r[6,8] = r[8,6] = np.sqrt(1-a**2)/2
    return r/b

# --- (2) Random bound entangled: Tiles UPB dressed with Haar local unitaries ---
_e = np.eye(3, dtype=complex)
_n = lambda v: v/np.linalg.norm(v)
_tiles = [np.kron(_e[0], _n(_e[0]-_e[1])),
          np.kron(_e[2], _n(_e[1]-_e[2])),
          np.kron(_n(_e[0]-_e[1]), _e[2]),
          np.kron(_n(_e[1]-_e[2]), _e[0]),
          np.kron(_n(_e[0]+_e[1]+_e[2]), _n(_e[0]+_e[1]+_e[2]))]

def random_bound_entangled():
    U = np.kron(unitary_group.rvs(3, random_state=rng),
                unitary_group.rvs(3, random_state=rng))
    P = sum(np.outer(U@p, (U@p).conj()) for p in _tiles)
    s = is_ppt((np.eye(9) - P)/4.0)
    print(f"is ppt {s}")
    return s

In [39]:
m = 3
nb_identified = 0
for i in tqdm(range(10)):
    state = random_bound_entangled()
    obj, omega_sym = SDP(antiSymProjector(2), state, m,m)
    if obj > SDP_TOL:
        nb_identified += 1
    # print(obj)
print(f"Correctly identified {nb_identified} of 10 in C3xC3")

  0%|          | 0/10 [00:00<?, ?it/s]

Correctly identified 10 of 10 in C3xC3
